## Resources
Here is the complete, modular code structure for the Day 5 curriculum, optimized for immediate execution inside a Google Colab notebook.

### 🔧 Cell 1: Install Dependencies
Run this cell first to set up ChromaDB (vector database), Sentence Transformers (local embedding generation), and the Groq client.

In [ ]:
# Install the necessary libraries quietly
!pip install -q groq chromadb sentence-transformers openai
print("✅ Core dependencies successfully installed!")

✅ Core dependencies successfully installed!


### 🔑 Cell 2: API Keys & Environment Configuration
This cell extracts your Openrouter API key safely from Colab's built-in Secrets manager.

💡 Before Running: Click the 🔑 (Secrets) icon on the far left panel of your Colab window, add a new secret named exactly `OPENROUTER_API_KEY`, paste your Openrouter key into the value field, and switch the "Notebook Access" toggle to ON.

In [ ]:
import os
from google.colab import userdata

# Safely extract key from Colab Secrets manager
try:
    # We'll fetch 'OPENROUTER_API_KEY' from secrets and store it in an environment variable.
    OPENROUTER_API_KEY_VALUE = userdata.get("OPENROUTER_API_KEY")
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY_VALUE # Use a more generic env variable name
    print("✅ Openrouter API key successfully verified and loaded.")
except Exception as e:
    print("❌ Error loading key. Ensure 'OPENROUTER_API_KEY' is active in your Colab Secrets menu.")
    print(f"Details: {e}")

✅ Openrouter API key successfully verified and loaded.


### 🔢 Cell 3: Intuition Lab — Vector Similarity Demo
Run this cell to observe semantic search in action using a lightweight local embedding model. Notice how high the similarity score remains when words share a meaning despite being completely unique spellings.

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load a compact, fast open-source embedding model (~80MB)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Generate meaning arrays (vectors) for sentences
vector_puppy = embedder.encode("I love my puppy")
vector_dog   = embedder.encode("My dog is adorable")
vector_taxes = embedder.encode("Please file your taxes")

# Calculate cosine similarities
score_similar   = float(util.cos_sim(vector_puppy, vector_dog))
score_different = float(util.cos_sim(vector_puppy, vector_taxes))

print("--- Vector Similarity Intuition Lab ---")
print(f"Similarity ('puppy' vs 'dog')  : {round(score_similar, 2)}  (High similarity - related concepts)")
print(f"Similarity ('puppy' vs 'taxes'): {round(score_different, 2)} (Low similarity - unrelated concepts)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- Vector Similarity Intuition Lab ---
Similarity ('puppy' vs 'dog')  : 0.72  (High similarity - related concepts)
Similarity ('puppy' vs 'taxes'): 0.11 (Low similarity - unrelated concepts)


### 🗄️ Cell 4: Knowledge Base Ingestion & Vector Storage
This cell sets up an in-memory instance of ChromaDB, populates it with unstructured corporate facts, and automatically generates meaning-vectors behind the scenes.

In [ ]:
import chromadb

# Fictional knowledge corpus chunks
acme_knowledge_base = [
    "Acme Corp offers 24 days of paid annual leave to all full-time employees.",
    "Employees can work from home up to 3 days per week with manager approval.",
    "The office is located at 42 MG Road, Bangalore, and opens at 9:00 AM.",
    "Acme Corp reimburses internet bills up to 1000 rupees per month for remote staff.",
    "New employees are on probation for the first 6 months of employment.",
    "The annual company retreat happens every December in Goa.",
]

# Initialize local client
chroma_client = chromadb.Client()

# Create or reload a vector storage partition
collection = chroma_client.get_or_create_collection(name="acme_handbook")

# Load text chunks into vector database (Chroma automatically applies default embeddings)
collection.add(
    documents=acme_knowledge_base,
    ids=[f"doc_chunk_{i}" for i in range(len(acme_knowledge_base))]
)

print(f"✅ Success: Vector DB populated with {collection.count()} validated chunks.")

✅ Success: Vector DB populated with 6 validated chunks.


### 🤖 Cell 5: Building the Core RAG Pipeline Engine
This cell builds the complete operational bridge. It accepts an arbitrary human question, extracts semantic matches from ChromaDB, builds an explicit prompt sandbox, and lets Groq generate a grounded response.

In [ ]:
from openai import OpenAI # Openrouter uses the OpenAI client library

# Initialize Openrouter client
# Set the base_url to Openrouter's endpoint
# Use the OPENROUTER_API_KEY from the environment variable
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY")
)
# Use an Openrouter-compatible model name.
MODEL_NAME = "poolside/laguna-m.1:free"

def generate_rag_response(user_question: str, max_chunks: int = 2) -> str:
    # Step 1: RETRIEVE closest contextual matches from the database
    db_results = collection.query(query_texts=[user_question], n_results=max_chunks)
    retrieved_text_blocks = db_results["documents"][0]

    # Merge blocks into a single target string context window
    context_window = "\n".join(retrieved_text_blocks)

    # Step 2: AUGMENT prompt structure with specific constraints
    grounded_prompt = f"""You are a strict internal company assistant. Answer the question using ONLY the factual context provided below.\nIf the explicit answer is missing or cannot be directly inferred from the context, respond with exactly: "I don't have that information."

Context:
{context_window}

Question: {user_question}
Answer:"""

    # Step 3: GENERATE finalized textual answer from the LLM
    api_completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": grounded_prompt}],
        temperature=0.0  # Zero out creativity to minimize hallucination risks
    )

    return api_completion.choices[0].message.content

# Test semantic capability immediately using alternate words
print("--- Operational Pipeline Check ---")
print("Query: 'How many holidays do I get?'")
print(f"Answer: {generate_rag_response('How many holidays do I get?')}")

--- Operational Pipeline Check ---
Query: 'How many holidays do I get?'
Answer: 
24 days of paid annual leave.



### 📖 Cell 6: Execution Loop — The Interactive Bot
Run this cell to spin up an interactive chat instance right inside your notebook terminal to experiment with content matches and strict boundary rules.

In [ ]:
print("📖 Acme Corporate Knowledge Bot active. [Type 'quit' or 'exit' to stop]")
print("-" * 65)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"quit", "exit", "q"}:
        print("👋 Session closed. Good luck with your enterprise AI applications!")
        break
    if not user_input.strip():
        continue

    bot_reply = generate_rag_response(user_input)
    print(f"Bot: {bot_reply}\n")

📖 Acme Corporate Knowledge Bot active. [Type 'quit' or 'exit' to stop]
-----------------------------------------------------------------
You: How many days of leave do I get?
Bot: 
Answer: 24 days of paid annual leave.


You: quit
👋 Session closed. Good luck with your enterprise AI applications!


### ✂️ Cell 7: Document Chunking Lab
This final utility component implements a text-slicing function using a sliding-window calculation to preserve context across boundaries.

In [ ]:
def dynamic_text_chunker(source_document: str, window_size: int = 80, window_overlap: int = 20) -> list:
    """Slices a long string into character-based segments containing structured text overlap."""
    generated_chunks = []
    cursor_position = 0
    document_length = len(source_document)

    while cursor_position < document_length:
        end_position = cursor_position + window_size
        chunk_slice = source_document[cursor_position:end_position]
        generated_chunks.append(chunk_slice)
        # Shift cursor forward by window size minus specified overlap
        cursor_position = end_position - window_overlap

    return generated_chunks

# Sample narrative text block
sample_corporate_history = (
    "Acme Corp was founded in 2010 in Bangalore. "
    "It builds weather-prediction software for farmers. "
    "The company has 200 employees across three offices. "
    "Its flagship product, RainSense, launched in 2018. "
    "Acme was awarded Best AgriTech Startup in 2021."
)

print("--- Text Processing & Chunking Lab ---")
processed_slices = dynamic_text_chunker(sample_corporate_history, window_size=90, window_overlap=25)

for index, text_slice in enumerate(processed_slices):
    print(f"Chunk Segment {index}: {repr(text_slice)}")

--- Text Processing & Chunking Lab ---
Chunk Segment 0: 'Acme Corp was founded in 2010 in Bangalore. It builds weather-prediction software for farm'
Chunk Segment 1: 'diction software for farmers. The company has 200 employees across three offices. Its flag'
Chunk Segment 2: 's three offices. Its flagship product, RainSense, launched in 2018. Acme was awarded Best '
Chunk Segment 3: '8. Acme was awarded Best AgriTech Startup in 2021.'
